# Evaluation of Fuzzy Method / Model

expliquer en vif le projet 

## Data

TODO : deux tableaux avec des réponses à nos questions (profile et travel), adapter les réponses pour avoir des recommendation différentes à la fin 

TODO : un excel avec plusieurs destinations (maybe que focus sur l'europe pour l'instant) avec des scores qui correspondent aux valeurs crisp des réponses

Présenter les tableaux et expliquer ce qu'on attend comme résultats pour les utilisateurs

Expliquation des tableaux :
* donner les topics
* donner les questions
* donner les priority et leur justifications (pour profile et pour travel)

(potentiellement juste donner les pages du pdf pour les questions, topics et priorities puis justifier en text)


Expliquer les résultats en se basant sur les valeurs des destinations -> se baser sur les valeurs des destinations pour biaiser les réponse fictives des questionnaires

### Data Preprocessing

Adapter les valeurs fuzzy en crisp pour les tableaux des questionnaires

Faire en sorte que ça s'adapte aussi pour les teste avec un user pour l'évaluation en "User Testing"

### 1.

In [2]:
%pip install numpy pandas

  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl (12.6 MB)
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB 325.1 kB/s eta 0:00:34
   ---------------------------------------- 0.1/11.0 MB 901.1 kB/s eta 0:00:13
   - -------------------------------------- 0.4/11.0 MB 2.3 MB/s eta 0:00:05
   --- ------------------------------------ 1.0/11.0 MB 5.0 MB/s eta 0:00:02
   ------------ --------------------------- 3.3/11.0 MB 12.5 MB/s eta 0:00:01
   ------------------------------ --------- 8.3/11.0 MB 26.6 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 65.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd

# Load our dataset
cities = pd.read_csv("Cities.csv")

# Themes with at least one questionnaire question
selected_themes = [
    1, 2, 3, 4, 6, 7, 8, 12, 13, 14,
    17, 18, 19, 20, 21, 22, 23, 24, 27
]

selected_columns = [
    f"t{theme}" for theme in selected_themes
]

# We keep only relevant criteria
cities_selected = cities[
    ["City"] + selected_columns
].copy()

print("Number of destinations:", len(cities_selected))
print("Number of themes:", len(selected_columns))
print("Missing values:", cities_selected.isna().sum().sum())

display(cities_selected.head())


Number of destinations: 41
Number of themes: 19
Missing values: 0


,City,t1,t2,t3,t4,t6,t7,t8,t12,t13,t14,t17,t18,t19,t20,t21,t22,t23,t24,t27
0,Amsterdam,6,6,3,2,4,9,6,6,2,4,4,1,4,6,4,2,9,3,4
1,Barcelona,6,9,3,2,6,6,4,6,4,4,4,2,3,4,4,6,6,3,4
2,Berlin,9,6,4,3,6,9,6,9,2,4,6,3,4,6,4,2,9,4,6
3,Bratitslava,3,3,4,4,3,4,4,3,3,4,4,6,4,4,3,3,4,2,6
4,Bruge,2,4,6,4,6,4,6,2,4,3,3,2,4,4,4,3,3,2,6


## 2. questionnaire input types

In [4]:

# Questionnaire input types for each selected theme

question_types = {
    "t1":  {"P": None, "T": "scale_7"},
    "t2":  {"P": "slider_100", "T": None},
    "t3":  {"P": "categorical", "T": None},
    "t4":  {"P": "slider_100", "T": "slider_100"},
    "t6":  {"P": None, "T": "scale_7"},
    "t7":  {"P": "scale_7", "T": None},
    "t8":  {"P": "scale_7", "T": None},
    "t12": {"P": "slider_100", "T": "scale_7"},
    "t13": {"P": "slider_100", "T": "multiple_choice"},
    "t14": {"P": "scale_7", "T": "scale_7"},
    "t17": {"P": "scale_7", "T": None},
    "t18": {"P": "scale_7", "T": "budget"},
    "t19": {"P": "scale_7", "T": None},
    "t20": {"P": "scale_7", "T": None},
    "t21": {"P": "scale_7", "T": "scale_5"},
    "t22": {"P": "categorical", "T": "categorical"},
    "t23": {"P": "scale_7", "T": "multiple_choice"},
    "t24": {"P": "scale_7", "T": "scale_5"},
    "t27": {"P": None, "T": "multiple_choice"}
}

# Check if all selected themes are represented
assert set(question_types) == set(selected_columns)

print("Number of themes:", len(question_types))

print(
    "Personality questions:",
    sum(q["P"] is not None for q in question_types.values())
)

print(
    "Trip questions:",
    sum(q["T"] is not None for q in question_types.values())
)


Number of themes: 19
Personality questions: 16
Trip questions: 12


### 3. Reference answer

In [5]:

from copy import deepcopy

# Default Personality Test answers
base_personality = {
    "t2": 50,                  # Gastronomy (0-100)
    "t3": "Combination",      # Authenticity
    "t4": 50,                  # Relaxation (0-100%)
    "t7": 4,                   # Infrastructure (1-7)
    "t8": 4,                   # Hygiene precautions (1-7)
    "t12": 50,                 # Unique experiences (0-100)
    "t13": 50,                 # Natural landscapes (0-100)
    "t14": 4,                  # Travel pace (1-7)
    "t17": 4,                  # Flexibility (1-7)
    "t18": 4,                  # Price sensitivity (1-7)
    "t19": 6,                  # Personal safety (1-7)
    "t20": 6,                  # Healthcare (1-7)
    "t21": 4,                  # Accommodation comfort (1-7)
    "t22": "Mild",             # Climate preference
    "t23": 4,                  # Transport effort tolerance (1-7)
    "t24": 4                   # Physical activity (1-7)
}

# Default Trip Questionnaire answers
base_trip = {
    "t1": 4,                   # Change of scenery (1-7)
    "t4": 50,                  # Relaxation (0-100%)
    "t6": 4,                   # Landmarks (1-7)
    "t12": 4,                  # Unique experiences (1-7)
    "t13": ["Mountains", "Beaches"],
    "t14": 4,                  # Travel pace (1-7)
    "t18": {
        "preferred": 1800,
        "maximum": 2500
    },                        # Budget in CHF
    "t21": 3,                  # Accommodation comfort (1-5)
    "t22": "Mild",             # Climate
    "t23": ["Walking", "Public transport"],
    "t24": 3,                  # Physical activity (1-5)
    "t27": ["Quality time"]
}


### 4. Initialize four profiles


Here we created four snthetic taveler profiles to evaluate our fuzzy recommendation system.
- Adventure lover
- Relaxation lover
- Food Explorer
- Balanced Traveler

Each traveler answers the questions from the personality test and trip questionnaire.
We use 19 themes. The goal is to check whether our system produces logical and consistent destination recommendations for different traveler preferences.

In [6]:

# Initialize the four scenarios
scenario_names = [
    "Adventure Lover",
    "Relaxation Lover",
    "Food Explorer",
    "Balanced Traveler"
]

personality_answers = {
    name: deepcopy(base_personality)
    for name in scenario_names
}

trip_answers = {
    name: deepcopy(base_trip)
    for name in scenario_names
}

# Scenario 1: Adventure Lover
personality_answers["Adventure Lover"].update({
    "t2": 35,
    "t3": "Local life",
    "t4": 15,
    "t7": 6,
    "t12": 90,
    "t13": 90,
    "t14": 7,
    "t17": 6,
    "t21": 3,
    "t22": "Cool",
    "t23": 6,
    "t24": 7
})

trip_answers["Adventure Lover"].update({
    "t1": 7,
    "t4": 10,
    "t6": 3,
    "t12": 7,
    "t13": ["Mountains", "Forests"],
    "t14": 7,
    "t18": {"preferred": 1600, "maximum": 2200},
    "t21": 2,
    "t22": "Mild",
    "t23": ["Walking", "Bicycle"],
    "t24": 5,
    "t27": ["Group activities"]
})

# Scenario 2: Relaxation Lover
personality_answers["Relaxation Lover"].update({
    "t2": 40,
    "t4": 85,
    "t7": 2,
    "t8": 2,
    "t12": 25,
    "t13": 80,
    "t14": 1,
    "t17": 2,
    "t21": 7,
    "t22": "Warm",
    "t23": 2,
    "t24": 1
})

trip_answers["Relaxation Lover"].update({
    "t1": 3,
    "t4": 90,
    "t6": 2,
    "t12": 2,
    "t13": ["Beaches", "Lakes"],
    "t14": 1,
    "t18": {"preferred": 2000, "maximum": 3000},
    "t21": 5,
    "t22": "Warm",
    "t23": ["Car", "Public transport"],
    "t24": 1,
    "t27": ["Quality time"]
})

# Scenario 3: Food Explorer
personality_answers["Food Explorer"].update({
    "t2": 95,
    "t3": "Local life",
    "t4": 35,
    "t12": 75,
    "t13": 45,
    "t14": 5,
    "t17": 5,
    "t18": 5,
    "t22": "Warm",
    "t24": 3
})

trip_answers["Food Explorer"].update({
    "t1": 6,
    "t4": 30,
    "t6": 6,
    "t12": 6,
    "t13": ["Countryside"],
    "t14": 5,
    "t18": {"preferred": 2200, "maximum": 3000},
    "t22": "Warm",
    "t23": ["Walking", "Public transport"],
    "t24": 2,
    "t27": ["Meeting new people", "Quality time"]
})

print("Synthetic profiles created successfully!")
print("Number of scenarios:", len(scenario_names))


Synthetic profiles created successfully!
Number of scenarios: 4


### 5. Additional test scenarios


To further evaluate the fuzzy recommendation system, we added three scenarios:

- **Adventurer Seeking Relaxation:** A traveler who usually enjoys adventure but wants a relaxing vacation this time.
- **Budget-Constrained Traveler:** A traveler with a limited maximum budget.
- **Adventure Preference Change:** A traveler whose desired adventure level changes while all other preferences remain the same.

These scenarios help us test whether the system adapts to changing trip preferences, respects budget constraints, and behaves consistently when a single input is modified.


In [7]:

from copy import deepcopy

# Scenario 5: Adventurer Seeking Relaxation
name = "Adventurer Seeking Relaxation"

personality_answers[name] = deepcopy(
    personality_answers["Adventure Lover"]
)

trip_answers[name] = deepcopy(
    trip_answers["Adventure Lover"]
)

# only change current trip preferences
trip_answers[name].update({
    "t4": 90,     # High relaxation
    "t14": 1,     # Very relaxed travel pace
    "t24": 1      # Very low physical activity
})


# Scenario 6: Budget-Constrained Traveler
name = "Budget-Constrained Traveler"

personality_answers[name] = deepcopy(
    personality_answers["Balanced Traveler"]
)

trip_answers[name] = deepcopy(
    trip_answers["Balanced Traveler"]
)

# only change the budget
trip_answers[name]["t18"] = {
    "preferred": 500,
    "maximum": 700
}


# Scenario 7: Adventure Preference Change
name = "Adventure Preference Change"

personality_answers[name] = deepcopy(
    personality_answers["Adventure Lover"]
)

trip_answers[name] = deepcopy(
    trip_answers["Adventure Lover"]
)

# Here we change only one variable
trip_answers[name]["t24"] = 1


# Update the list of scenarios
scenario_names = [
    "Adventure Lover",
    "Relaxation Lover",
    "Food Explorer",
    "Balanced Traveler",
    "Adventurer Seeking Relaxation",
    "Budget-Constrained Traveler",
    "Adventure Preference Change"
]

print("Total number of scenarios:", len(scenario_names))


Total number of scenarios: 7


In [8]:
# Expected themes from the questionnaires
expected_p = {
    theme for theme, q in question_types.items()
    if q["P"] is not None
}

expected_t = {
    theme for theme, q in question_types.items()
    if q["T"] is not None
}


# Verify the seven scenarios
assert len(scenario_names) == 7

for name in scenario_names:
    assert set(personality_answers[name]) == expected_p
    assert set(trip_answers[name]) == expected_t

# Scenario 5: Same personality, different trip
assert (
    personality_answers["Adventure Lover"]
    == personality_answers["Adventurer Seeking Relaxation"]
)

# Scenario 6: Only the budget changes
budget_test = deepcopy(
    trip_answers["Budget-Constrained Traveler"]
)
budget_test["t18"] = deepcopy(
    trip_answers["Balanced Traveler"]["t18"]
)

assert budget_test == trip_answers["Balanced Traveler"]

# Scenario 7: Only the adventure preference changes
adventure_test = deepcopy(
    trip_answers["Adventure Preference Change"]
)
adventure_test["t24"] = trip_answers[
    "Adventure Lover"
]["t24"]

assert adventure_test == trip_answers["Adventure Lover"]

print("All seven scenarios are correctly configured!")


All seven scenarios are correctly configured!


### 6. Display Answers


In [9]:
# Then we check all scenarios
for name in scenario_names:
    p = personality_answers[name]
    t = trip_answers[name]

    assert set(p) == expected_p, (
        f"Personality questions mismatch: {name}"
    )
    assert set(t) == expected_t, (
        f"Trip questions mismatch: {name}"
    )

    assert t["t18"]["preferred"] <= t["t18"]["maximum"]

    print(
        f"{name}: "
        f"{len(p)} personality answers, "
        f"{len(t)} trip answers"
    )


Adventure Lover: 16 personality answers, 12 trip answers
Relaxation Lover: 16 personality answers, 12 trip answers
Food Explorer: 16 personality answers, 12 trip answers
Balanced Traveler: 16 personality answers, 12 trip answers
Adventurer Seeking Relaxation: 16 personality answers, 12 trip answers
Budget-Constrained Traveler: 16 personality answers, 12 trip answers
Adventure Preference Change: 16 personality answers, 12 trip answers


In [10]:
# Display for Personality Test responses
personality_df = pd.DataFrame.from_dict(
    personality_answers,
    orient="index"
)

display(personality_df)

# Display for Trip Questionnaire responses
trip_df = pd.DataFrame.from_dict(
    trip_answers,
    orient="index"
)

display(trip_df)


,t2,t3,t4,t7,t8,t12,t13,t14,t17,t18,t19,t20,t21,t22,t23,t24
Adventure Lover,35,Local life,15,6,4,90,90,7,6,4,6,6,3,Cool,6,7
Relaxation Lover,40,Combination,85,2,2,25,80,1,2,4,6,6,7,Warm,2,1
Food Explorer,95,Local life,35,4,4,75,45,5,5,5,6,6,4,Warm,4,3
Balanced Traveler,50,Combination,50,4,4,50,50,4,4,4,6,6,4,Mild,4,4
Adventurer Seeking Relaxation,35,Local life,15,6,4,90,90,7,6,4,6,6,3,Cool,6,7
Budget-Constrained Traveler,50,Combination,50,4,4,50,50,4,4,4,6,6,4,Mild,4,4
Adventure Preference Change,35,Local life,15,6,4,90,90,7,6,4,6,6,3,Cool,6,7


,t1,t4,t6,t12,t13,t14,t18,t21,t22,t23,t24,t27
Adventure Lover,7,10,3,7,"[Mountains, Forests]",7,"{'preferred': 1600, 'maximum': 2200}",2,Mild,"[Walking, Bicycle]",5,[Group activities]
Relaxation Lover,3,90,2,2,"[Beaches, Lakes]",1,"{'preferred': 2000, 'maximum': 3000}",5,Warm,"[Car, Public transport]",1,[Quality time]
Food Explorer,6,30,6,6,[Countryside],5,"{'preferred': 2200, 'maximum': 3000}",3,Warm,"[Walking, Public transport]",2,"[Meeting new people, Quality time]"
Balanced Traveler,4,50,4,4,"[Mountains, Beaches]",4,"{'preferred': 1800, 'maximum': 2500}",3,Mild,"[Walking, Public transport]",3,[Quality time]
Adventurer Seeking Relaxation,7,90,3,7,"[Mountains, Forests]",1,"{'preferred': 1600, 'maximum': 2200}",2,Mild,"[Walking, Bicycle]",1,[Group activities]
Budget-Constrained Traveler,4,50,4,4,"[Mountains, Beaches]",4,"{'preferred': 500, 'maximum': 700}",3,Mild,"[Walking, Public transport]",3,[Quality time]
Adventure Preference Change,7,10,3,7,"[Mountains, Forests]",7,"{'preferred': 1600, 'maximum': 2200}",2,Mild,"[Walking, Bicycle]",1,[Group activities]


# Model

In [ ]:
print("hello world")

hello world


Observations